
# One-Rec — embed shard 0/5

Pure embedding worker for the catalog-expansion queue: fresh preview URL per
track (they expire in ~15 min), Discogs-EffNet embeddings via a **packed
batch-64** runner (equivalence-gated against the reference implementation,
with automatic fallback), npz checkpoints. Takes every 5th queue row
starting at index 0.


In [ ]:
import asyncio, glob, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd

%pip install -q essentia-tensorflow==2.1b6.dev1389 aiohttp
!wget -q -nc https://essentia.upf.edu/models/feature-extractors/discogs-effnet/discogs-effnet-bs64-1.pb
print("model:", os.path.getsize("discogs-effnet-bs64-1.pb") / 1e6, "MB")
T_SESSION = time.time()

CFG = dict(
    shard=0,
    n_shards=5,
    shard_start=0,
    deezer_rps=8,
    batch=512,            # download/embed unit, inside the preview-URL TTL
    ckpt_every=2_000,
    mel_workers=4,
    time_budget_h=11.2,
    seed=42,
)
WORK = Path("/kaggle/working")
time.sleep(CFG["shard"] * 7)  # stagger shard start-up against shared rate limits


In [ ]:
def input_glob(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

queue = pd.read_parquet(input_glob("embed_queue.parquet")[0])

done_ids = set()
for f in input_glob("ext_ckpt_*.npz") + sorted(str(p) for p in WORK.glob("ext_ckpt_*.npz")):
    z = np.load(f, allow_pickle=True)
    done_ids.update(z["ids"].tolist())
miss_ids = set()
for p in input_glob("ext_misses*.parquet") + sorted(str(x) for x in WORK.glob("ext_misses*.parquet")):
    miss_ids |= set(pd.read_parquet(p)["id"])

tail = queue.iloc[CFG["shard_start"]:].reset_index(drop=True)
mine = tail[np.arange(len(tail)) % CFG["n_shards"] == CFG["shard"]]
todo = mine[~mine["id"].isin(done_ids) & ~mine["id"].isin(miss_ids)].reset_index(drop=True)
print(f"queue {len(queue):,} | shard rows {len(mine):,} | already done {len(done_ids):,} "
      f"| known misses {len(miss_ids):,} | todo {len(todo):,}")


## Deezer client — fresh URL per track; only permanent misses are recorded

In [ ]:
import aiohttp

HDRS = {"User-Agent": "one-rec-research/1.0"}


class RateLimiter:
    def __init__(self, rps):
        self.min_int = 1.0 / rps
        self.next_t = 0.0
        self.lock = asyncio.Lock()

    async def acquire(self):
        async with self.lock:
            now = time.monotonic()
            wait = self.next_t - now
            self.next_t = max(now, self.next_t) + self.min_int
        if wait > 0:
            await asyncio.sleep(wait)


DEEZER = RateLimiter(CFG["deezer_rps"])


async def deezer_get(session, url, params=None, retries=6):
    for attempt in range(retries):
        await DEEZER.acquire()
        try:
            async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=15)) as resp:
                data = await resp.json(content_type=None)
        except Exception:
            await asyncio.sleep(2 * (attempt + 1))
            continue
        if isinstance(data, dict) and data.get("error", {}).get("code") == 4:
            await asyncio.sleep(5 + 3 * attempt)  # shared-IP quota pressure: back off harder
            continue
        return data
    return None


DL_SEM = asyncio.Semaphore(16)


async def fetch_blob(session, dzid):
    """(blob, permanent_miss). Rate-limit/timeouts are TRANSIENT — never
    recorded as misses, the track just stays queued for the next pass."""
    d = await deezer_get(session, f"https://api.deezer.com/track/{dzid}")
    if d is None:
        return None, False                      # transient: API unreachable/quota
    if not isinstance(d, dict) or not d.get("preview"):
        return None, True                       # permanent: no preview on Deezer
    async with DL_SEM:
        for _ in range(3):
            try:
                async with session.get(d["preview"], timeout=aiohttp.ClientTimeout(total=30)) as resp:
                    if resp.status == 200:
                        return await resp.read(), False
            except Exception:
                pass
            await asyncio.sleep(1)
    return None, False                          # transient: CDN hiccup


async def _download_batch(rows):
    async with aiohttp.ClientSession(headers=HDRS) as session:
        out = await asyncio.gather(*(fetch_blob(session, d) for d in rows["deezer_id"]))
    return [(i, blob, perm) for i, (blob, perm) in zip(rows["id"], out)]


def download_batch(rows):
    # Callable from anywhere: Jupyter's main thread already runs an event
    # loop (asyncio.run would raise), so hop to a private thread there.
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(_download_batch(rows))
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, _download_batch(rows)).result()


## Embedder — packing INSIDE single-threaded fork workers

The v2 shards deadlocked by running TensorFlow in the MAIN process alongside
forked TF workers. Proven process model restored: the main process never
touches TF; each worker owns one single-threaded TF context and packs patches
across its own chunk of clips into full batch-64 forwards. The equivalence
gate runs inside a worker too, with a hard timeout — a hang now fails the run
visibly instead of zombieing for hours.

In [ ]:
import multiprocessing as mp
import tempfile

_ref_model = _mel = _packed = None


def _init_worker():
    global _ref_model, _mel, _packed
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
    os.environ["TF_NUM_INTEROP_THREADS"] = "1"
    import essentia
    essentia.log.warningActive = False   # silence the "No network created" flood
    from essentia.standard import (TensorflowInputMusiCNN, TensorflowPredict,
                                   TensorflowPredictEffnetDiscogs)
    _ref_model = TensorflowPredictEffnetDiscogs(graphFilename="discogs-effnet-bs64-1.pb",
                                                output="PartitionedCall:1")
    _mel = TensorflowInputMusiCNN()
    _packed = TensorflowPredict(graphFilename="discogs-effnet-bs64-1.pb",
                                inputs=["serving_default_melspectrogram"],
                                outputs=["PartitionedCall:1"])


def _decode(blob):
    f = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    try:
        f.write(blob)
        f.close()
        from essentia.standard import MonoLoader
        return MonoLoader(filename=f.name, sampleRate=16000, resampleQuality=4)()
    finally:
        os.unlink(f.name)


def _melbands(audio):
    from essentia.standard import FrameGenerator
    return np.array([_mel(fr) for fr in FrameGenerator(audio, frameSize=512, hopSize=256,
                                                       startFromZero=True)], np.float32)


def _patches(bands, hop, tail):
    P = 128
    n_full = 1 + (len(bands) - P) // hop if len(bands) >= P else 0
    out = [bands[i * hop:i * hop + P] for i in range(n_full)]
    if tail != "discard":
        rest = bands[n_full * hop:] if n_full else bands
        if len(rest) >= 16:
            if tail == "zeros":
                pad = np.zeros((P - len(rest), bands.shape[1]), np.float32)
                out.append(np.concatenate([rest, pad]))
            elif tail == "edge":
                out.append(np.concatenate([rest, np.repeat(rest[-1:], P - len(rest), axis=0)]))
            else:  # cycle
                reps = int(np.ceil(P / len(rest)))
                out.append(np.tile(rest, (reps, 1))[:P])
    return out


def _run64(plist, owners=None):
    """Full batch-64 forwards over a patch list; returns (n_patches, 1280)."""
    from essentia import Pool
    outs = []
    for lo in range(0, len(plist), 64):
        chunk = plist[lo:lo + 64]
        batch = np.zeros((64, 128, 96), np.float32)
        batch[:len(chunk)] = np.stack(chunk)
        pool = Pool()
        pool.set("melspectrogram", batch[:, None, :, :])
        outs.append(np.asarray(_packed(pool)["PartitionedCall:1"]).reshape(64, -1)[:len(chunk)])
    return np.concatenate(outs) if outs else np.zeros((0, 1280), np.float32)


def _embed_chunk(args):
    """One worker, one chunk of clips. mode None = reference path; (hop, tail)
    = packed with cross-clip batch-64 packing."""
    mode, clips = args
    if mode is None:
        out = []
        for tid, blob in clips:
            try:
                audio = _decode(blob)
                if len(audio) < 16000:
                    continue
                out.append((tid, _ref_model(audio).mean(axis=0).astype(np.float32)))
            except Exception:
                pass
        return out
    hop, tail = mode
    owners, plist = [], []
    for tid, blob in clips:
        try:
            audio = _decode(blob)
            if len(audio) < 16000:
                continue
            for pt in _patches(_melbands(audio), hop, tail):
                owners.append(tid)
                plist.append(pt)
        except Exception:
            pass
    if not plist:
        return []
    embs = _run64(plist)
    sums, counts = {}, {}
    for tid, e in zip(owners, embs):
        sums[tid] = sums.get(tid, 0.0) + e.astype(np.float64)
        counts[tid] = counts.get(tid, 0) + 1
    return [(tid, (sums[tid] / counts[tid]).astype(np.float32)) for tid in dict.fromkeys(owners)]


def _gate_probe(clips):
    """Runs INSIDE one worker: derive (hop, tail) whose patch counts match the
    reference on real clips, then require mean-vec cosine > 0.999."""
    ref, mels = {}, {}
    for tid, blob in clips:
        try:
            audio = _decode(blob)
            if len(audio) < 16000:
                continue
            ref[tid] = _ref_model(audio)
            mels[tid] = _melbands(audio)
        except Exception:
            pass
    if len(ref) < 3:
        return None
    for hop in (128, 96, 64, 62, 32):
        for tail in ("discard", "zeros", "edge", "cycle"):
            if any(len(_patches(mels[t], hop, tail)) != len(ref[t]) for t in ref):
                continue
            cos = []
            for t in ref:
                pm = _run64(_patches(mels[t], hop, tail)).mean(axis=0)
                rm = ref[t].mean(axis=0)
                cos.append(float(pm @ rm / ((np.linalg.norm(pm) * np.linalg.norm(rm)) or 1.0)))
            if min(cos) > 0.999:
                return hop, tail, min(cos)
    return None


## Gate — worker-side equivalence probe with a hard timeout; the main
process never touches TensorFlow

In [ ]:
pool = mp.get_context("fork").Pool(CFG["mel_workers"], initializer=_init_worker)

MODE = None   # None = proven reference path; (hop, tail) = packed
if len(todo):
    sample = download_batch(todo.head(8))
    sample_ok = [(tid, blob) for tid, blob, _ in sample if blob][:5]
    if len(sample_ok) >= 3:
        try:
            verdict = pool.apply_async(_gate_probe, (sample_ok,)).get(timeout=900)
        except mp.TimeoutError:
            print("gate TIMED OUT — rebuilding pool, using the reference path", flush=True)
            pool.terminate(); pool.join()
            pool = mp.get_context("fork").Pool(CFG["mel_workers"], initializer=_init_worker)
            verdict = None
        if verdict:
            MODE = (verdict[0], verdict[1])
            print(f"PACKED path ON (hop={verdict[0]}, tail={verdict[1]}, cos={verdict[2]:.5f})", flush=True)
        else:
            print("PACKED equivalence not established — reference path", flush=True)


def embed_batch(payloads):
    if not payloads:
        return []
    n = CFG["mel_workers"]
    chunks = [payloads[i::n] for i in range(n)]
    handles = [pool.apply_async(_embed_chunk, ((MODE, c),)) for c in chunks if c]
    out = []
    for h in handles:
        out.extend(h.get(timeout=1800))   # watchdog: a hung worker ERRORS the run fast
    return out


## Embed loop — per-shard checkpoints, time-budgeted

In [ ]:
from concurrent.futures import ThreadPoolExecutor

SHARD_TAG = f"s{CFG['shard']}"
pending_ids, pending_embs, new_misses = [], [], []
emb_total = 0


def flush(force=False):
    global pending_ids, pending_embs
    if pending_ids and (force or len(pending_ids) >= CFG["ckpt_every"]):
        n = len(list(WORK.glob(f"ext_ckpt_{SHARD_TAG}_*.npz")))
        np.savez(WORK / f"ext_ckpt_{SHARD_TAG}_{n:03d}.npz",
                 ids=np.array(pending_ids, dtype=object),
                 embs=np.stack(pending_embs).astype(np.float16))
        pending_ids, pending_embs = [], []
    if new_misses:
        pd.DataFrame({"id": sorted(set(new_misses))}).to_parquet(
            WORK / f"ext_misses_{SHARD_TAG}.parquet", index=False)


batches = [todo.iloc[i:i + CFG["batch"]] for i in range(0, len(todo), CFG["batch"])]
throughput_printed = False
with ThreadPoolExecutor(max_workers=1) as fetcher:
    future = fetcher.submit(download_batch, batches[0]) if batches else None
    for bi in range(len(batches)):
        triples = future.result()
        if bi + 1 < len(batches):
            future = fetcher.submit(download_batch, batches[bi + 1])
        payloads = [(i, b) for i, b, _ in triples if b]
        new_misses.extend(i for i, b, perm in triples if not b and perm)
        t0 = time.time()
        embedded = embed_batch(payloads)
        for tid, emb in embedded:
            pending_ids.append(tid)
            pending_embs.append(emb)
        emb_total += len(embedded)
        if not throughput_printed and payloads:
            rate = len(payloads) / max(time.time() - t0, 1e-9)
            print(f"embed throughput: {rate:.2f} clips/s "
                  f"({'packed' if MODE is not None else 'reference'} path)", flush=True)
            throughput_printed = True
        flush()
        if bi % 10 == 0:
            print(f"batch {bi + 1}/{len(batches)} | embedded {emb_total:,} | "
                  f"perm-misses {len(set(new_misses))} | "
                  f"{(time.time() - T_SESSION) / 3600:.1f}h", flush=True)
        if (time.time() - T_SESSION) / 3600 > CFG["time_budget_h"]:
            print(f"time budget reached at batch {bi + 1} — stopping cleanly")
            break
flush(force=True)
print(f"shard {CFG['shard']} done: {emb_total:,} embedded this session")
print("files:", sorted(f.name for f in WORK.glob("ext_*")))
